In [ ]:
# Cell 1 — self-contained bootstrap (same pattern as Step 2)
import torch, os, sys

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

GITHUB_TOKEN = ""  # paste for this session only
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Evenjlin/deep-space-interference-ml.git"

if not os.path.exists('/content/deep-space-interference-ml'):
    !git clone {REPO_URL} /content/deep-space-interference-ml
else:
    !cd /content/deep-space-interference-ml && git pull

%cd /content/deep-space-interference-ml
!pip install -r requirements.txt -q

sys.path.insert(0, os.getcwd())

In [ ]:
# Cell 2 — imports + config for Fig.3's condition
import numpy as np
from sklearn.metrics import roc_auc_score
from src.channel import SignalConfig, mix_signal, generate_soi, generate_noise
from src.model import estimate_covariance, mahalanobis_score, fit_pca_detector, pca_score

# OUR ASSUMPTION: fd_max=0.0 reproduces Fig.3's "no Frequency Shift" condition
cfg = SignalConfig(fd_max=0.0)
rng = np.random.default_rng(42)

N_SYMBOLS = 16   # OUR ASSUMPTION: detection window length (not specified in paper)
SIR_DB = 20.0    # BASE-PAPER FACT (Fig.3 caption)
SNR_DB = 15.0    # BASE-PAPER FACT (Fig.3 caption)

In [ ]:
# Cell 3 — build interference-free training pool, fit C and PCA
def make_clean_sample(cfg, n_symbols, snr_db, rng):
    s, bits, fd, phi_m = generate_soi(cfg, n_symbols, rng)
    w = generate_noise(len(s), rng)
    rho_snr = 10 ** (snr_db / 10)
    return s + w / np.sqrt(rho_snr)

M_TRAIN = 2000  # sized so covariance dim (2*16*8=256) is well-estimated
clean_samples = [make_clean_sample(cfg, N_SYMBOLS, SNR_DB, rng) for _ in range(M_TRAIN)]

C = estimate_covariance(clean_samples)
C_inv = np.linalg.inv(C)
pca = fit_pca_detector(clean_samples)
print("Covariance shape:", C.shape, " | PCA components kept:", pca.n_components_)

In [ ]:
# Cell 4 — build labeled test sets and score both detectors
def build_test_set(cfg, n_symbols, rng, interference_type, n_test, sir_db, snr_db):
    samples, labels = [], []
    for _ in range(n_test):
        d = mix_signal(cfg, n_symbols, rng, interference_type=interference_type,
                        contamination=0.5,  # BASE-PAPER FACT: test contamination = 50%
                        sir_db_range=(sir_db, sir_db), snr_db_range=(snr_db, snr_db),
                        apply_shift=False)
        samples.append(d["x"]); labels.append(d["label"])
    return samples, np.array(labels, dtype=int)

results = {}
# OUR ASSUMPTION: "AWGN" in Fig.3/Table1 = our "fawgn" (Filtered AWGN)
for itype in ["tone", "chirp", "fawgn"]:
    samples, labels = build_test_set(cfg, N_SYMBOLS, rng, itype, 200, SIR_DB, SNR_DB)
    maha_scores = [mahalanobis_score(x, C_inv) for x in samples]
    pca_scores  = [pca_score(x, pca) for x in samples]
    results[itype] = {
        "mahalanobis": roc_auc_score(labels, maha_scores),
        "pca": roc_auc_score(labels, pca_scores),
    }
    print(f"{itype:6s}  Mahalanobis AUC={results[itype]['mahalanobis']:.4f}  "
          f"PCA AUC={results[itype]['pca']:.4f}")

In [ ]:
# Cell 5 — reproducibility comparison vs paper's Table 1 (Modulation:No, Random Shift:No row)
import pandas as pd
import matplotlib.pyplot as plt

paper_mahalanobis = {"tone": 0.9861, "chirp": 0.9955, "fawgn": 0.9954}  # BASE-PAPER FACT

df = pd.DataFrame(results).T
comparison = pd.DataFrame({
    "paper_mahalanobis_auc": paper_mahalanobis,
    "our_mahalanobis_auc": df["mahalanobis"],
})
comparison["difference"] = comparison["our_mahalanobis_auc"] - comparison["paper_mahalanobis_auc"]
print(comparison)

df.plot(kind="bar", figsize=(6,4))
plt.ylabel("AUC"); plt.axhline(1.0, linestyle="--", color="gray")
plt.title("Our reproduction of Fig.3 (Mahalanobis vs PCA)\n[Autoencoder bar pending - Step 4]")
plt.tight_layout()
plt.savefig("figures/fig3_reproduction_partial.png", dpi=150)
plt.show()

os.makedirs("results", exist_ok=True)
comparison.to_csv("results/mahalanobis_reproducibility_table.csv")
df.to_csv("results/fig3_partial_auc.csv")